# Processing in JPYN

In [ ]:
import os
import scanpy as sc
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import sys
import os
sys.path.append('C:/Users/Riss Kellogg/miniconda3/envs/scvi-env/Lib/site-packages')
import doubletdetection
import scvi

# Working with denoised.h5 files output by CellBender. See how much background RNA was removed using metrics.csv file

In [ ]:
os.listdir(path="C:/Users/Riss Kellogg/Desktop/CellBenderMetricsFiles")

In [ ]:
pd.read_csv("C:/Users/Riss Kellogg/Desktop/CellBenderMetricsFiles/Ctrl1denoised_metrics.csv")

In [ ]:
#Create a dataframe from metrics data
path = "C:/Users/Riss Kellogg/Desktop/CellBenderMetricsFiles/"
metrics = []

for file in [x for x in os.listdir(path) if x.endswith('metrics.csv')]:
    t= pd.read_csv(path + file,
                  header = None, names = ['Metric', 'Value']).set_index('Metric').T
    t['File'] = file
    metrics.append(t)
    
metrics = pd.concat(metrics).reset_index()

In [ ]:
metrics
#1 row per sample
#Notice that Fraction_counts_removed are all high, especially for zaps (>10%)

In [ ]:
#Histogram of fraction_counts_removed

metrics.plot.bar(x = 'File', y = 'fraction_counts_removed_from_cells', grid=False)

#6 samples total

from IPython.display import clear_output
clear_output(wait=True)

# Working with new filtered.h5 files

In [ ]:
os.listdir(path="C:/Users/Riss Kellogg/Desktop/CellBenderCleanH5Files")

In [ ]:
pathH5s = "C:/Users/Riss Kellogg/Desktop/CellBenderCleanH5Files/"
adatas = [x for x in os.listdir(pathH5s) if x.endswith('filtered.h5')]
#you could also use a whitelist of barcodes from the filtered barcodes for each sample
#in this case, you would open the unfiltered then filter it based on the whitelist

In [ ]:
adatas

In [ ]:
def load_it(adata):
    samp = adata.split('_')[0]
    dx = adata.split('_')[1]
    adata = sc.read_10x_h5(pathH5s + adata)
    adata.obs['Sample'] = samp
    #adata.obs['DX'] = dx
    #adata.obs['Sample'] = adata.obs['Patient'] + '_' + adata.obs['DX']
    adata.obs.index = adata.obs.index + '-' + samp
    return adata

In [ ]:
adatas = [load_it(ad) for ad in adatas]

In [ ]:
adatas

In [ ]:
#background RNA is now gone
#Cellbender is good at removing backgoround, but not so good at calling cells
#Look at the adata objects above: they are all cells x genes. We expected 10,000 cells, and they are all 16-20k+. So cellbender has included too many.
def qc(adata):
    sc.pp.filter_cells(adata, min_genes = 200) #anything below 200 genes is very likely not going to be a cell
    adata.var["mt"] = adata.var_names.str.startswith('mrp143') #try filtering mitochondrial genes in SP. need to pull mito genes and make a separate list, then pass LOC names. here is just one as an example.
    adata.var["ribo"] = adata.var_names.str.startswith("RPS", "RPL") #placeholder for ribosomal genes
    sc.pp.calculate_qc_metrics(adata, qc_vars=['mt','ribo'], inplace=True, percent_top=[20], log1p=True)
    remove = ['total_counts_mt', 'log1p_total_counts_mt', 'total_counts_ribo', 
          'log1p_total_counts_ribo', 'total_counts_hb', 'log1p_total_counts_hb'] #get rid of unuseful columns
    
    adata.obs = adata.obs[[x for x in adata.obs.columns if x not in remove]]
    adata.var_names_make_unique()
    
    return adata

In [ ]:
adatas = [qc(ad) for ad in adatas]

In [ ]:
df = pd.concat([x.obs for x in adatas])
df = df.sort_values('Sample')

In [ ]:
df

In [ ]:
#Visualize how many total counts were in each sample using Ridge Plots
#look at the commented-out "#value" below. Switch through them to see each dataset. 
#Red line is the median.

#value = "pct_counts_mt"
#value = "n_genes" #some may have multi-modal distributions; usually the first bump is filtered out
#value = 'pct_counts_in_top_20_genes' #genes that are highly expressed all the time, they account for a big proportion of the total reads. Just 20 genes account for 20% of the total counts from a cell. 
value = "log1p_total_counts" #total counts, but on log1p scale.

sns.set(style="white", rc={"axes.facecolor": (0, 0, 0, 0)})

g = sns.FacetGrid(df, row="Sample", hue="Sample", aspect=15, height=0.5, palette="tab20")

g.map(sns.kdeplot, value, clip_on=False, fill=True, alpha=1, linewidth=1.5)
g.map(sns.kdeplot, value, clip_on=False, color="w", lw=2)

g.map(plt.axhline, y=0, lw=2, clip_on=False)

def label(x, color, label):
    ax = plt.gca()
    ax.text(0, .2, label, fontweight="bold", color=color,
            ha="left", va="center", transform=ax.transAxes)

g.map(label, value)

g.figure.subplots_adjust(hspace=-.6)

g.set_titles("")
g.set(yticks=[], ylabel="")
g.despine(bottom=True, left=True)

for ax in g.axes.flat:
    ax.axvline(x=df[value].median(), color='r', linestyle='-')


plt.show()

from IPython.display import clear_output
clear_output(wait=True)

# Pre-processing

In [ ]:
#Filter out low quality cells based on QC metrics and get rid of doublets
#Get rid of doublets using doubletdetection. SCVI tools is ok too but takes longer.
import doubletdetection
from scipy.stats import median_abs_deviation as mad
import numpy as np

In [ ]:
#Look at one specific sample
a = df[df.Sample == 'Zap4'].log1p_total_counts #save log1p_total_counts to "a"

In [ ]:
a

In [ ]:
#get the median 
np.median(a)

In [ ]:
np.median(a) - 5 * mad(a) #median - 5 deviations below

In [ ]:
np.median(a) + 5 * mad(a) #median + 5 deviations above

In [ ]:
#plot the distribution

ax=sns.displot(a)
plt.axvline(np.median(a) - 5 * mad(a))
plt.axvline(np.median(a) + 5 * mad(a))
%matplotlib inline
plt.show()

from IPython.display import clear_output
clear_output(wait=True)

In [ ]:
# We see that all the majority of cells lie between the 2 lines are within 5 absolute deviations of the median 
# Reccd by Germain et al 2020 (Genome Biology https://doi.org/10.1186/s13059-020-02136-7)

In [ ]:
#Detect outliers using MAD (median absolute deviation)
#Helper function to flag cells as being in or out of mad bounds

def mad_outlier(adata, metric, nmads, upper_only = False): #metric is log1p_total_counts
    M = adata.obs[metric]
    
    if not upper_only:
        return (M < np.median(M) - nmads * mad(M)) | (M > np.median(M) +nmads * mad(M))
    
    return (M > np.median(M) + nmads + mad(M))

In [ ]:
#Recc'd settings for DoubletDetection
clf = doubletdetection.BoostClassifier(
    n_iters=10,
    clustering_algorithm="louvain",
    standard_scaling = True,
    pseudocount = 0.1,
    n_jobs=-1)

In [ ]:
#Create the pre-processing (pp) function

def pp(adata):
    adata = adata[adata.obs.pct_counts_mt < 25] #you can lower this based on the overal distribution of your dataset
    
    bool_vector = mad_outlier(adata, 'log1p_total_counts', 5) +\
            mad_outlier(adata, 'log1p_n_genes_by_counts', 5) +\
            mad_outlier(adata, 'pct_counts_in_top_20_genes', 5) +\
            mad_outlier(adata, 'pct_counts_mt', 3, upper_only = True)
    adata = adata[~bool_vector]

    adata.uns['cells_removed'] = sum(bool_vector)

    doublets = clf.fit(adata.X).predict(p_thresh=1e-16, voter_thresh=0.5) #recc thresholds
    doublet_score = clf.doublet_score()

    adata.obs["doublet"] = doublets
    adata.obs["doublet_score"] = doublet_score

    adata.uns['doublets_removed'] = adata.obs.doublet.sum()
    adata = adata[adata.obs.doublet == 0]
    adata.var_names_make_unique()

    return adata

In [ ]:
#Call the preprocessing (pp) function on all adatas
adatas = [pp(ad) for ad in adatas] #create a list of individual adata objects

In [ ]:
#Look at #of cells | cells removed | doublets removed for each sample
for adata in adatas:
    print(len(adata), adata.uns['cells_removed'], adata.uns['doublets_removed'])

These stats look good. Note that if your data are more noisy (ie, it says 0 doublets removed; thats not correct) you will need to adjust settings.

If the number of cells removed in MAD filtering is very large relative to overall cells, it may be worthwhile to increase thresholds or use a less conservative method. ~5-15% is typical.

Expected doublet rate is typically ~2-10% depending on the number of cells sequenced for 10x workflows

If anything deviates drastically, visit that sample specifically.

You can try to increase the p_thresh= (1e-16 to start) closer to 1 (1e-3) or move voter_thresh closer to 0.

****MAKE SURE PREPROCESSING LOOKS RIGHT BECAUSE IT WILL AFFECT EVERYTHING DOWNSTREAM*****

single cell best practices by the Theis lab is a good resource https://www.nature.com/articles/s41576-023-00586-w https://www.sc-best-practices.org/preprocessing_visualization/quality_control.html

In [ ]:
#Look at how distributions look, after this preprocessing
df2 = pd.concat([x.obs for x in adatas])
df2 = df2.sort_values('Sample')

In [ ]:
#value = "pct_counts_mt"
#value = "n_genes"
#value = 'pct_counts_in_top_20_genes'
value = "log1p_total_counts"

sns.set(style="white", rc={"axes.facecolor": (0, 0, 0, 0)})

g = sns.FacetGrid(df2, row="Sample", hue="Sample", aspect=15, height=0.5, palette="tab20")

g.map(sns.kdeplot, value, clip_on=False, fill=True, alpha=1, linewidth=1.5)
g.map(sns.kdeplot, value, clip_on=False, color="w", lw=2)

g.map(plt.axhline, y=0, lw=2, clip_on=False)

def label(x, color, label):
    ax = plt.gca()
    ax.text(0, .2, label, fontweight="bold", color=color,
            ha="left", va="center", transform=ax.transAxes)

g.map(label, value)

g.figure.subplots_adjust(hspace=-.6)

g.set_titles("")
g.set(yticks=[], ylabel="")
g.despine(bottom=True, left=True)

for ax in g.axes.flat:
    ax.axvline(x=df2[value].median(), color='r', linestyle='-')


plt.show()

from IPython.display import clear_output
clear_output(wait=True)

# END OF PREPROCESSING
ambient RNA, doublets, and low quality cells are now removed.

# Saving all adatas into one concatenated adata

In [ ]:
import scanpy as sc
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scvi

In [ ]:
#%pip install celltypist hyperopt "ray[tune]" anndata2ri

In [ ]:
import celltypist
from celltypist import models

In [ ]:
#models.get_all_models()

In [ ]:
adatas

In [ ]:
#combine all preprocessed adata objects into 1 adata object
adata = sc.concat(adatas)

In [ ]:
adata

# ADATA Save Point #1

In [ ]:
#ADATA SAVE POINT 1: save as a h5ad format file
adata.write_h5ad('6_27_24_cellbenderdatasintegrated.h5ad')

In [ ]:
##############################################3

# Part II: Mapping and Cluster Labelling

In [ ]:
#%pip install scvi-tools
#need a version with AutoTune included

In [ ]:
#Load the concatentated adata object. Note that this is NOT "INTEGRATED", all weve done is put the data into 1 object.
adata = sc.read_h5ad('6_27_24_cellbenderdatasintegrated.h5ad')

In [ ]:
adata #remember that this is cells x genes

In [ ]:
# Use SCVI to get the embeddings to get the data to integrate together 
# 2024 single-cell guide Part 2: time = 21:43 
# You could use SCVI default parameters or Use hyperparameter tuning to make the model as good as possible
#from scvi.autotune import ModelTuner
from scvi.autotune import ModelTuner
from ray import tune

In [ ]:
#Filter out lowly-expressed genes
sc.pp.filter_genes(adata, min_cells = 50)

In [ ]:
adata
#now we're down to 18,173 genes

In [ ]:
# here is where you could narrow these 18k genes down to the 2000 most differentially expressed genes

In [ ]:
#Model for hyperparameter tuning
model_cls = scvi.model.SCVI
model_cls.setup_anndata(adata, categorical_covariate_keys = ['Sample'],
                             continuous_covariate_keys=['pct_counts_mt', 'pct_counts_ribo'])

tuner = ModelTuner(model_cls)

In [ ]:
tuner.info()

In [ ]:
#^ a list of all the hyperparameters in our model that we can change. SCVI default is ok but we can tune for an even better job.
#5/13/24 tuner isnt working, so im using SCVI default params rather than searching for the best one.


In [ ]:
# search_space = {
#     "n_hidden": tune.choice([92, 128, 192, 256]),
#     "n_latent": tune.choice([10, 20, 30, 40, 50, 60]),
#     "n_layers": tune.choice([1, 2, 3]),
#     "lr": tune.loguniform(1e-4, 1e-2),
#     "gene_likelihood": tune.choice(["nb", "zinb"])}

In [ ]:
# results = tuner.fit(adata, metric="validation_loss",
#                     search_space = search_space,
#                    num_samples = 100, #how many combinations it will test; higher= more accurate but will take longer. training the model 100 times.
#                    max_epochs = 20)

# Run Integration Model (scvi)

In [ ]:
scvi.model.SCVI.setup_anndata(adata,
                              categorical_covariate_keys = ['Sample'],
                             continuous_covariate_keys=['pct_counts_mt', 'pct_counts_ribo'])

In [ ]:
#model if you want to pass tunable parameters
#model = scvi.model.SCVI(adata, n_hidden = 256, n_latent = 10, n_layers = 3, gene_likelihood = 'zinb')

In [ ]:
#USING THE DEFAULT SCVI MODEL PARAMETERS
model = scvi.model.SCVI(adata)

In [ ]:
model.train(early_stopping = True)
#max_epochs=200
#be aware: this takes 24hrs without a gpu

# SAVE THE MODEL

In [ ]:
model.save('6_27_24_cellbender_Ctrl134_Zap134_adatasintegrated.model')

# Get embeddings

In [ ]:
# Get the embeddings (used to find neighbors and do clutering). Using the model as an integration method
adata.obsm['X_scVI'] = model.get_latent_representation()
adata.obsm['X_scVI'].shape

# Calculate neighbors

In [ ]:
#calculate neighbors
sc.pp.neighbors(adata, use_rep = 'X_scVI')

# Do Leiden Clustering (Resolution = 1)

In [ ]:
#do clustering
sc.tl.leiden(adata, resolution = 1)

# Visualize using UMAP

In [ ]:
#visuzalize using UMAP
sc.tl.umap(adata)

In [ ]:
sc.pl.umap(adata, color =['leiden'], palette='tab20', frameon = True, legend_loc = "on data")

In [ ]:
sc.pl.umap(adata, color =['leiden'], palette='tab20', frameon = True, legend_loc = "on data")

# Normalize and Scale Data for UMAP visualization and celltype assignment, but always keep raw data in a different layer for DE

In [ ]:
#make a new layer called "counts" by copying the raw X data. this is saved for later DE.
adata.layers['counts'] = adata.X.copy()

In [ ]:
#normalizing: the right way to do it is the default method by scanpy : according to nature methods "Comparison of transformations for single-cell RNAseq data")
sc.pp.normalize_total(adata)
sc.pp.log1p(adata)

#SO: Raw data is stored in adata.layers['counts']. It was a copy of adata.X before adata.X was normalized.
#Adata.X was normalized and then log1p transformed. Log1p means take the log of (1+data).

# ADATA SAVEPOINT #2

In [ ]:
#UPDATE ADATA
adata.write_h5ad('6_27_24_cellbenderclean_adata_norm_scaled.h5ad')

In [ ]:
adata

In [ ]:
adata.uns.markers

# Assign Condition

In [ ]:
#Add the column "Condition" to adata.obs
def map_condition(x):
    if 'Zap' in x:
        return 'UVB-dosed'
    else:
        return 'Control'

adata.obs['condition'] = adata.obs.Sample.map(map_condition) #give a new column to adata.obs called condition, labels 'Control' or 'UVB dosed'


In [ ]:
adata.obs

# Plot the UMAP showing distribution regarding Condition

In [ ]:
fig, ax = plt.subplots()

sc.pl.umap(adata, color=['condition'], frameon=True, title='', show=False, ax=ax, palette='bwr')
ax.set_xticks(range(-4,22,2))
ax.set_yticks(range(-8,17,2))
ax.legend(loc='upper left')
matplotlib.pyplot.savefig('6_28_24_UMAP_Condition_Plot.png', dpi='figure')
plt.show()

# Plot the UMAP showing all Leiden Clusters

In [ ]:
fig, ax = plt.subplots()
%matplotlib inline
sc.pl.umap(adata, color=['leiden'], frameon=True, legend_loc = 'on data', title='', show=False, ax=ax, palette='tab20')
ax.set_xticks(range(-4,22,2))
ax.set_yticks(range(-8,17,2))
matplotlib.pyplot.savefig('C:/Users/Riss Kellogg/Desktop/scRNAseq Results C134 Z134 6_28_24/INDIVIDUAL FIGURES/Fig1 300dpi/UMAP_AllClusters_Plot_600dpi.png', dpi=600)
plt.show()

# Prepare the Dictionary of Cell Types

In [ ]:
#Dictionary- map cell type to cluster. simple groups: phagocytes, red cells, colorless cells, vibratile cells.
cell_type = {"0":"Phagocytes",
"1":"Phagocytes",
"2":"Phagocytes",
"3":"Phagocytes",
"4":"Phagocytes",
"5":"Phagocytes", #NOTE: not confident; shows only prolif markers, no TLRs, no c3. phago based on map alone.
"6":"Phagocytes",
"7":"Vibratile Cells",
"8":"Phagocytes",
"9":"Phagocytes",
"10":"Colorless Cells",
"11":"Phagocytes",
"12":"Phagocytes",
"13":"Red Cells",
"14":"Colorless Cells",
"15":"Phagocytes",
"16":"Red Cells",
"17":"Phagocytes",
"18":"Phagocytes",
"19":"Vibratile Cells",
"20":"Phagocytes",
"21": "Vibratile Cells" #NOTE: not confident; shows only dynein. Assigned vib based on map position.
    }

adata.obs['cell_type'] = adata.obs.leiden.map(cell_type)

# To assign cell types, look for marker genes of phagocytes, red cells, colorless cells, and vibratile cells

In [ ]:
#note: this is a quick way to do DE to find marker genes, but not reccd for actual DE analysis.
sc.tl.rank_genes_groups(adata, groupby = 'leiden')
markers = sc.get.rank_genes_groups_df(adata, group = None)
markers = markers[(markers.pvals_adj < 0.05) & (markers.logfoldchanges > 0.5)]
adata.uns['markers'] = markers

In [ ]:
adata.uns['markers'].names

# ADATA SAVE POINT

In [ ]:
adata

In [ ]:
#UPDATE ADATA
adata.write_h5ad('6_27_24_cellbenderclean_adata_norm_scaled.h5ad')

# Map LOC name to Gene name using a lookup table dictionary

In [ ]:
import pandas as pd

#Read in the mapping excel file
excel_file = 'SpurpLOClookuptable.xlsx'
name_mapping_df = pd.read_excel(excel_file)

In [ ]:
name_mapping_df

In [ ]:
#Create the dictionary from that excel file above
name_mapping_dict = dict(zip(name_mapping_df['Gene_symbol'], name_mapping_df['Gene_name']))

In [ ]:
#name_mapping_dict

In [ ]:
#add these new names to the "markers" dataframe
markers['new_names'] = markers['names'].apply(lambda x: name_mapping_df.loc[name_mapping_df['Gene_symbol']==x, 'Gene_name'].values[0] if x in name_mapping_df['Gene_symbol'].values else x)

In [ ]:
markers

# Look for (now annotated) Marker Genes

# Red Cells

In [ ]:
#RED CELLS
#Start with cell types you know are going to be in the data
#LOC588806 is sp-pks1, the only marker gene i know (for red cells) IN DATASET
#LOC115922158 = sp-pks2, not in dataset
#LOC592147 = sp-pks2 (?another one), not in dataset
#sc.pl.umap(adata, color = ['LOC588806'], frameon= False, layer = 'scvi_normalized')

In [ ]:
markers[markers.names == 'LOC588806']

In [ ]:
markers[markers.names == 'LOC115922158']

In [ ]:
markers[markers.names == 'LOC592147']

# Phagocytes

In [ ]:
#SpTrf phagoytes
markers[markers.names == '185/333']

In [ ]:
#Sp-B7L3 phagocytes
markers[markers.names == 'LOC762504'] 

In [ ]:
#TLR phagocytes
markers[markers.names == 'Tlr1.1'] #TLR1.1

In [ ]:
markers[markers.names == 'LOC115929085'] #TLR2

In [ ]:
#TLR phagocytes
markers[markers.names == 'LOC105442321'] #TLR3

In [ ]:
markers[markers.names == 'LOC105446117'] #TLR4

In [ ]:
markers[markers.names == 'LOC115929919'] #TlR5

In [ ]:
markers[markers.names == 'LOC115926210'] #TLR13

In [ ]:
markers[markers.names == 'LOC100890666'] #Tollo

In [ ]:
#Cells with complement c3 (note: all phagocyte clusters)
markers[markers.names == 'c3']

# Vibratile Cells

In [ ]:
markers[markers.names == 'LOC591536'] #Sp-P2rx4

In [ ]:
markers[markers.names == 'FoxJ1']

In [ ]:
# Cells doing clotting (vibratile cells)
markers[markers.names == 'AMAS4']

In [ ]:
#Check for dynein (to indicate flagella = vibs)
markers[markers.names == 'LOC762758']

# Colorless Cells

In [ ]:
#Cells with bactericidal LOC100889613 (Assigned cluster as "bactericidal colorless cells")
markers[markers.names == 'LOC100889613']

In [ ]:
#DD104, DD104 protein, upregulated upon bacterial challenge and trauma
markers[markers.names == 'DD104']

In [ ]:
#LOC587263 lysozyme
#LOC753728 strongylocin2
#Going to call these "cytotoxic colorless cells"
markers[markers.names == 'LOC587263']

In [ ]:
markers[markers.names == 'LOC753728'] #strongylocin2

# DNA Repair Enzymes

In [ ]:
#Find XRCC1 DNA repair protein
markers[markers.names == 'LOC580530']

In [ ]:
#UV genes
#UVrag LOC586516
#UV RAD23 LOC578143
#sc.pl.umap(adata, color = ['LOC586516','LOC578143'], frameon= False, layer = 'scvi_normalized')
markers[markers.names == 'LOC578143'] #RAD23

In [ ]:
markers[markers.names == 'LOC586516'] #UVrag

# Markers of Proliferation (dividing cells)

In [ ]:
#Check for proliferation markers
markers[markers.names == 'LOC765052'] #PCNA; cell division. proliferating cell nuclear antigen

In [ ]:
#Check for proliferation markers
markers[markers.names == 'LOC100890576'] #PCNA; cell division. PCNA-associated factor.

In [ ]:
markers[markers.names == 'LOC115918456'] #PCNA; cell division. proliferating cell nuclear antigen-like

In [ ]:
markers[markers.names == 'LOC579977'] #histone H3, embryonic

In [ ]:
markers[markers.names == 'LOC373245'] #cyclin D

In [ ]:
markers[markers.names == 'LOC110980132'] #cyclin E

In [ ]:
markers[markers.names == 'ccne1'] #cyclin E

In [ ]:
markers[markers.names == 'LOC110984714'] #cyclin B

In [ ]:
markers[markers.names == 'ccnb2'] #cyclin B

In [ ]:
markers[markers.names == 'LOC110980811'] #TERT

In [ ]:
markers[markers.names == 'tert'] #TERT

In [ ]:
markers[markers.names == 'myc'] #MyC

# Stem-ness Markers

In [ ]:
#Check for stem-cell markers (piwi and vasa)
markers[markers.names == 'piwil1'] #piwi1

In [ ]:
markers[markers.names == 'LOC583253'] #piwi2; gene name is seali

In [ ]:
markers[markers.names == 'seali'] #piwi2; gene name is seali

In [ ]:
markers[markers.names == 'LOC373434'] #piwi3

In [ ]:
markers[markers.names == 'ddx4'] #Sp-vasa

In [ ]:
markers[markers.names == 'LOC110980149'] #Sp-vasa 

In [ ]:
#Check for Kruppel-like factors (Zn fingers); transcription factors essential for maintaining pluripotency

markers[markers.names == 'LOC373243'] #Sp-Kruppel

# Use Scanpy to plot a UMAP showing conditions

In [ ]:
fig, ax = plt.subplots()

sc.pl.umap(adata, color=['condition'], frameon=True, title='', show=False, ax=ax, palette='bwr')
ax.set_xticks(range(-4,22,2))
ax.set_yticks(range(-8,17,2))
ax.legend(loc='upper left')

matplotlib.pyplot.savefig('C:/Users/Riss Kellogg/Desktop/scRNAseq Results C134 Z134 6_28_24/INDIVIDUAL FIGURES/Fig1 300dpi/UMAP_Condition_600dpi.png', dpi=600)
plt.show()

# Use Scanpy to plot a UMAP showing distribution of assigned cell types

In [ ]:
fig, ax = plt.subplots()

sc.pl.umap(adata, color= 'cell_type', frameon = True, title='', show=False, ax=ax, palette= 'viridis')
ax.set_xticks(range(-4,22,2))
ax.set_yticks(range(-8,17,2))
ax.legend(loc='lower right', fontsize='9')
matplotlib.pyplot.savefig('C:/Users/Riss Kellogg/Desktop/scRNAseq Results C134 Z134 6_28_24/INDIVIDUAL FIGURES/Fig1 300dpi/UMAP_CellTypes_600dpi.png', dpi=600)
plt.show()

# Cell Counting: Plot using Matplotlib and Seaborn

In [ ]:
#Cell Counting: How many zap and control cells are in each cluster?
#Count # of cells in each cluster
cluster_tot_cells = adata.obs.groupby(['leiden']).count() 
cluster_tot_cells = dict(zip(cluster_tot_cells.index, cluster_tot_cells.doublet)) #gives a dictionary of sample:total cells
cluster_tot_cells


In [ ]:
#Group by Condition (ctrl vs zap),
cluster_type_counts = adata.obs.groupby(['leiden', 'condition']).count()
cluster_type_counts = cluster_type_counts[cluster_type_counts.sum(axis = 1) > 0].reset_index() #show only rows with at least 1 cell
cluster_type_counts = cluster_type_counts[cluster_type_counts.columns[0:3]] #keep only the first 4 columns

#cluster_type_counts.rename(columns = {'Sample':'Cells_in_condition'}, inplace = True)
cluster_type_counts.to_csv('fraction of ctrl and zap cells in each cluster.csv')

In [ ]:
#Add a new total_cells column
cluster_type_counts['total_cells'] = cluster_type_counts.Sample.map(cluster_tot_cells).astype(int) #cluster_tot_cells is the dictionary made above
cluster_type_counts['frequency'] = cellcluster_type_counts.Cells_in_condition / cell_type_counts.total_cells #add a frequency column (cells / total cells)
#cell_type_counts

In [ ]:
#CELL COUNTING
# "Condition" label was already assigned to each cell above
#Distinguish control samples from zapped samples
#adata.obs.Sample.unique().tolist()

In [ ]:
#For frequency, Count the number of cells in each cell type, AND the number of cells total in each sample
#Total cells in each sample
num_tot_cells = adata.obs.groupby(['Sample']).count() 
num_tot_cells = dict(zip(num_tot_cells.index, num_tot_cells.doublet)) #gives a dictionary of sample:total cells
num_tot_cells

In [ ]:
#Group by Sample, Condition (ctrl vs zap), and cell type NOTE: "doublet" column  is not really doublets, its total cells
cell_type_counts = adata.obs.groupby(['Sample', 'condition', 'cell_type']).count()
cell_type_counts = cell_type_counts[cell_type_counts.sum(axis = 1) > 0].reset_index() #show only rows with at least 1 cell
cell_type_counts = cell_type_counts[cell_type_counts.columns[0:4]] #keep only the first 4 columns

cell_type_counts.rename(columns = {'n_genes':'cell_type_count'}, inplace = True)
cell_type_counts

In [ ]:
#Add a new total_cells column
cell_type_counts['total_cells'] = cell_type_counts.Sample.map(num_tot_cells).astype(int) #num_tot_cells is the dictionary made above
cell_type_counts['frequency'] = cell_type_counts.cell_type_count / cell_type_counts.total_cells #add a frequency column (cells / total cells)
#cell_type_counts

In [ ]:
#Plot the frequency
import matplotlib.pyplot as plt
plt.figure(figsize = (8,6))

my_pal={'Control':'blue', 'UVB-dosed':'red'}
ax = sns.boxplot(data = cell_type_counts, x = 'cell_type', y = 'frequency', hue = 'condition', palette=my_pal)
ax2 = sns.swarmplot(data = cell_type_counts, x = 'cell_type', y = 'frequency', hue = 'condition', color = 'k', size = 6, dodge= True, legend = False)

for box in ax.artists:
    box.set_edgecolor('black')

#sns.move_legend(ax, "upper left")
#ax.legend_.texts[0].set_text('Control')
#ax.legend_.texts[1].set_text('Zap')

sns.move_legend(
    ax, "lower center",
    bbox_to_anchor=(.8, .8), ncol=4, title=None, frameon=False,
)

plt.ylabel('Frequency among all cells', fontsize=16, labelpad=10)
plt.xlabel('Cell Type', labelpad=10, fontsize= 16)
plt.xticks(rotation = 0, rotation_mode = 'anchor', fontsize = 16)
plt.yticks(np.arange(0,1.1,step=.2), fontsize=16)
plt.legend(loc='upper right', fontsize=16)

matplotlib.pyplot.savefig('C:/Users/Riss Kellogg/Desktop/scRNAseq Results C134 Z134 6_28_24/INDIVIDUAL FIGURES/Fig1 300dpi/BoxPlot_CellTypeFrequency_600dpi.png', dpi=600)
plt.show()


# SCVI Differential Expression Analysis, save output as dataframe

In [ ]:
adata

In [ ]:
ZvC_DE_genes = model.differential_expression(
     idx1=[(adata.obs.condition == 'UVB-dosed')],  # we use the same cells as chosen before
     idx2=[(adata.obs.condition == 'Control')]
)

In [ ]:
ZvC_DE_genes = ZvC_DE_genes[(ZvC_DE_genes['is_de_fdr_0.05'])] #filter so only DE genes are shown

In [ ]:
num_genes = df['gene_column'].nunique()
print(f'The number of unique genes in the DataFrame is: {num_genes}')


In [ ]:
#What is the maximum value lfc_mean value in this dataframe?

#ZvC_DE_genes.loc[ZvC_DE_genes['lfc_mean'].idxmax()]

In [ ]:
ZvC_DE_genes.loc[ZvC_DE_genes['lfc_mean'].idxmax()]

In [ ]:
# manipulate the DE results for plotting
ZvC_DE_genes["log10_pscore"] = np.log10(ZvC_DE_genes["proba_not_de"]) #get the log10p value
ZvC_DE_genes["negativelog10_pscore"] = (ZvC_DE_genes["log10_pscore"]*-1) #get the NEGATIVE log10p-value (what is finally plotted)
ZvC_DE_genes = ZvC_DE_genes.join(adata.var, how="inner")

In [ ]:
# #assign an Entrez ID column by copying the index column to it
ZvC_DE_genes["Gene_symbol"] = (ZvC_DE_genes.index)

In [ ]:
ZvC_DE_genes

In [ ]:
#Need to somehow map "gene name" to df.index... 
# # Load lookup table (csv)
lookup = pd.read_excel('SpurpLOClookuptable.xlsx', engine='openpyxl')

In [ ]:
lookup

In [ ]:
# #"lookup" is a pandas dataframe, and "ZvC_DE_genes" is a pandas dataframe.
# #Both contain the column "Gene_symbol", so should be able to merge on that. ZvC_DE_genes should end up with a new column "Gene_name"
ZvC_DE_genes = pd.merge(ZvC_DE_genes, lookup, on='Gene_symbol', how = 'inner')

In [ ]:
ZvC_DE_genes.head()

In [ ]:
ZvC_DE_genes.to_csv('RAWDATA_ZvC_DE_genes_VolcanoPlotInput_randomseed5623.csv')

In [ ]:
ZvC_DE_genes['Gene_ID'].nunique()

# Load in DEGs as a dataframe

In [ ]:
#Load in dataframe created previously by model.differnetial_expression

ZvC_DE_genes = pd.read_excel('C:/Users/Riss Kellogg/Desktop/scRNAseq Results C134 Z134 6_28_24/ZapAllDEGs_randomseed5623.xlsx', sheet_name='ALL')
#"C:\Users\Riss Kellogg\Desktop\scRNAseq Results C134 Z134 6_28_24\ZapAllDEGs_randomseed5623.xlsx"


In [ ]:
ZvC_DE_genes.head()

# Interactive Volcano plot of DE genes using scvi tools

# Bokeh Volcano Plot

In [ ]:
# The dataframe above has what we need to make a bokeh volcano plot: lfc_mean is X, -log10_pscore is Y.
# Import plotting tools
import pandas as pd
import bokeh
import selenium
from bokeh.io import push_notebook, show, output_notebook, export_png, export_svgs
from bokeh.layouts import row
from bokeh.plotting import figure, show, save
from bokeh.transform import factor_cmap
from bokeh.models import ColumnDataSource, Span
from bokeh.models.tools import HoverTool
from bokeh.resources import CDN
from bokeh.embed import file_html

In [ ]:
#Define some Functions

#A function that will allow us to add, color, and outline groups of differentially expressed genes
def add_points(p, df, x, y, color, alpha=0.2, outline=False): #alpha is opacity
    source1 = bokeh.models.ColumnDataSource(df)
    
    p.circle(x=x, y=y, size=7, alpha=alpha, source=source1, color=color, name='circles')
    
    if outline:
        p.circle(x=x, y=y, size=7, alpha=1, source=source1, color='black', fill_color=None, name='outlines')
        
    return p

#A function to separate significantly upregulated and downregulated genes
def selector(df):
    sig_up = (ZvC_DE_genes.negativelog10_pscore > 1.3) & (ZvC_DE_genes.lfc_mean > 1)
    sig_down = (ZvC_DE_genes.negativelog10_pscore > 1.3) & (ZvC_DE_genes.lfc_mean < (-1))
    ups = ZvC_DE_genes[sig_up]
    downs = ZvC_DE_genes[sig_down]
    return ups, downs

In [ ]:
from bokeh.plotting import figure, show
from bokeh.models import MathText

#Set all fonts in Bokeh to Arial


In [ ]:
#Create the volcano plot figure comparing Zaps vs. Controls DE genes (p)
p = bokeh.plotting.figure(x_range=[-6,6], y_range=[1.22,2], width=800, height=800)
p.xaxis.axis_label = 'Log2(Fold Change(UVB-Dosed:Control))'
p.yaxis.axis_label = '-log10(p value)'

#Add the hover tool
hover = HoverTool()
p.add_tools(hover)
hover.tooltips=[('Gene_ID', '@Gene_ID'),
                ('Name', '@Gene_name'),
                ('LOC', '@Gene_symbol'),
               ('-log10pval', '@negativelog10_pscore'),
               ('lfc', '@lfc_mean')]

#Add the pvalue cutoff of 0.05 line (note, when p=0.05, -log10(0.05) = 1.3)
p.line(x=[-10,10], y=[1.3,1.3],
      color='grey', line_dash='dashed', line_width=1)
from bokeh.models import Label
label = Label(x=-5, y=1.3, text="p = 0.05", text_font_size="16pt", text_color="black")
p.add_layout(label)

#Add the lfc <1 or lfc>1 cutoff lines
p.line(x=[-1,-1], y=[0,10],
      color='grey', line_dash='dashed', line_width=1)
p.line(x=[1,1], y=[0,10], 
      color='grey', line_dash='dashed', line_width=1)

#Add the data
p.circle('lfc_mean','negativelog10_pscore', source=ZvC_DE_genes, fill_alpha=0.1, size=10, color='grey')


#Color sig. upregulated genes in green; Color sig. downregulated genes in red
ups, downs = selector(ZvC_DE_genes)
p = add_points(p, ups, 'lfc_mean','negativelog10_pscore', color='green', alpha=0.6, outline=True)
p = add_points(p, downs, 'lfc_mean', 'negativelog10_pscore', color='red', alpha=0.6, outline=True)

#when using add_points to color ups and downs, it wont accept "legend_label" as an argument, 
#so redoing here so legend labels will show up
p.circle('lfc_mean','negativelog10_pscore', source=ups, color='green', alpha=0.6, 
         legend_label='Significantly upregulated in UVB-dosed cells')
p.circle('lfc_mean','negativelog10_pscore', source=downs, color='red', alpha=0.6, 
         legend_label='Significantly downregulated in UVB-dosed cells')


#Add the legend
p.legend.location = "top_right"
#p.legend.title = "Legend"
p.legend.border_line_color = "#000000"
p.legend.border_line_width = 3
p.legend.label_text_font_size = "16pt"
p.legend.label_text_color = "black"

#Adjust axis label settings
p.xaxis.axis_label_text_font_size = "16pt"
p.yaxis.axis_label_text_font_size = "16pt"
p.xaxis.major_label_text_font_size = "16pt"
p.yaxis.major_label_text_font_size = "16pt"
p.xaxis.axis_label_text_color = "black"
p.yaxis.axis_label_text_color = "black"
p.xaxis.axis_label_text_font_style = "normal" #no italics
p.yaxis.axis_label_text_font_style = "normal" #no italics

#Adjust tick label settings
p.xaxis.major_label_text_color = "black"
p.xaxis.major_tick_line_color = "black"
p.yaxis.major_label_text_color = "black"
p.yaxis.major_tick_line_color = "black"

#Keeps the created plot in-line in the notebook
#bokeh.io.reset_output()
#bokeh.io.output_notebook()


from bokeh.io import export_png
from bokeh.io.export import export_svgs
p.output_backend = "svg"

export_svgs(p, filename='C:/Users/Riss Kellogg/Desktop/scRNAseq Results C134 Z134 6_28_24/INDIVIDUAL FIGURES/Fig2 300dpi/AllCellsDEGs_LFC1_VolcanoPlot.svg')
#export_png(p, filename='C:/Users/Riss Kellogg/Desktop/scRNAseq Results C134 Z134 6_28_24/INDIVIDUAL FIGURES/Fig2 300dpi/AllCellsDEGs_LFC1_VolcanoPlot.png')
show(p)

#Green = Significantly upregulated in Zaps
#Red = Significantly downregulated in Zaps

In [ ]:
len(ups)

In [ ]:
len(downs)

In [ ]:
ups.to_csv('AllCellsUps.csv')

In [ ]:
downs.to_csv('AllCellsDowns.csv')

In [ ]:
# Open the image with Pillow and set the DPI
with Image.open("C:/Users/Riss Kellogg/Desktop/scRNAseq Results C134 Z134 6_28_24/INDIVIDUAL FIGURES/Fig2 300dpi/8_29_24_Zaps_DEGenes_LFC1_VolcanoPlot.png") as img:
    img.save("C:/Users/Riss Kellogg/Desktop/scRNAseq Results C134 Z134 6_28_24/INDIVIDUAL FIGURES/Fig2 300dpi/test.png", dpi=(300, 300))

In [ ]:
#pip install pillow

from PIL import Image

# GO (gene ontology) using GOATools

In [ ]:
# NOTE: this method will work with gene symbols or ENTREZ IDs. 
#If you have ensemble IDS youll need to make a dictionary to convert them.
# all seem to be in either gene symbol or ENTREZ format (LOCs are LOC+Entrez#)

In [ ]:
#%pip install goatools

In [ ]:
import goatools

#First need to make the background gene set
#First find out what the NCBI taxonomy ID is for the organism. For Spurp, its: 7668
#https://www.ncbi.nlm.nih.gov/datasets/taxonomy/7668/
#Go to http://www.ncbi.nlm.nih.gov/gene.
In the search bar, copy and paste this string:
"7668"[Taxonomy ID] AND alive[property] AND genetype protein coding[Properties]
Hit search
It gives you Items: 1 to 20 of 27525, so 27,525 coding genes.
Use Send To > File > Tabular text > Create file.
File will download.
Move the downloaded file (called gene_result.txt) to your current directory using:
!mv /C:/users/Riss Kellogg/Downloads/gene_result.txt . 

Now need to find where the .py script called "ncbi_gene_results_to_python.py" is
"C:\Users\Riss Kellogg\miniconda3\Scripts\ncbi_gene_results_to_python.py"
"C:\Users\Riss Kellogg\miniconda3\Lib\site-packages\goatools"

Below I am calling !python to the ncbi_gene_results_to_python.py, saying give me an output (-o) file w my chosen name, and passing it the downloaded ncbi gene list (gene_result.txt)

In [ ]:
!python "C:/Users/Riss Kellogg/miniconda3/Scripts/ncbi_gene_results_to_python.py" -o genes_ncbi_spurpuratus_proteincoding.py gene_result.txt 

end up with the new file, genes_ncbi_spurpuratus_proteincoding.py

In [ ]:
from genes_ncbi_spurpuratus_proteincoding import GENEID2NT as GeneID2nt_spurp

In [ ]:
#Import goatool packages
from goatools.base import download_go_basic_obo
from goatools.base import download_ncbi_associations
from goatools.obo_parser import GODag
from goatools.anno.genetogo_reader import Gene2GoReader
from goatools.goea.go_enrichment_ns import GOEnrichmentStudyNS

In [ ]:
#Download additional NCBI data
obo_fname = download_go_basic_obo()
fin_gene2go = download_ncbi_associations()
obodag = GODag("go-basic.obo")

In [ ]:
#obodag #a dictionary of different GO terms and their metadata

In [ ]:
#GeneID2nt_spurp[574826] #If you pass this an entrez geneID it will pull up the info for that gene

In [ ]:
#Since this data uses gene symbols and LOC names, need to make a new dictionary called mapper that related gene symbol to entrezID
mapper = {}

for key in GeneID2nt_spurp:
    mapper[GeneID2nt_spurp[key].Symbol] = GeneID2nt_spurp[key].GeneID
    
inv_map = {v: k for k, v in mapper.items()} #a reverse mapper that goes from entrez to symbol

In [ ]:
#mapper

In [ ]:
#Read NCBI gene2go. Store annotations in a list of named tuples.
objanno = Gene2GoReader(fin_gene2go, taxids=[7668])

#returns namespace2association where namespace is:
#     BP: biological process
#     MF: molecular_function
#     CC: cellular_components
#         association is a dictionary:
#                 key = NCBI GeneID
#                 value = a set of GO IDs associated with that gene
ns2assoc = objanno.get_ns2assc()    

In [ ]:
#ns2assoc #this is a dictionary of dictionaries. 

In [ ]:
goeaobj = GOEnrichmentStudyNS(
    GeneID2nt_spurp.keys(), #list of spurp protein coding genes
    ns2assoc, #geneid/GO associations
    obodag, #ontologies
    propagate_counts = False,
    alpha = 0.05, #default significance cutoff
    methods = ['fdr_bh']) #default multiple-test correction method

In [ ]:
#List of all the GO terms that contain duplicate GO terms
GO_items = []

temp = goeaobj.ns2objgoea['BP'].assoc
for item in temp:
    GO_items += temp[item]
    
temp = goeaobj.ns2objgoea['CC'].assoc
for item in temp:
    GO_items += temp[item]
    
temp = goeaobj.ns2objgoea['MF'].assoc
for item in temp:
    GO_items += temp[item]

# GO Enrichment: Pass in previous DE results dataframe, then using GOATOOLS to assign GO Terms. Looking at Zaps first.

In [ ]:
#Load in dataframe created previously by model.differnetial_expression

ZvC_DE_genes = pd.read_excel('C:/Users/Riss Kellogg/Desktop/scRNAseq Results C134 Z134 6_28_24/ZapAllDEGs_randomseed5623.xlsx', sheet_name='ALL')
#"C:\Users\Riss Kellogg\Desktop\scRNAseq Results C134 Z134 6_28_24\ZapAllDEGs_randomseed5623.xlsx"

In [ ]:
#Purpose is to go from Gene List to Enriched Pathways

#Load Differential Expression Analysis DF: 
scvi_de = ZvC_DE_genes

#Filter the output dataframe: keep only fdr 0.05=True, and lfc is >0.5.
scvi_de = scvi_de[(scvi_de['is_de_fdr_0.05']) & (scvi_de.lfc_mean > 0.5)]

scvi_de_upreg_genelist = scvi_de.Gene_symbol
scvi_de_upreg_genelist

In [ ]:
#A function that does the mapping from gene ID to GO, then GO analysis, then filters on pval, then parses output and makes a dataframe

def go_it(test_genes):
    print(f'input genes: {len(test_genes)}')
    
    mapped_genes = []
    for gene in test_genes:
        try:
            mapped_genes.append(mapper[gene])
        except:
            pass
    print(f'mapped genes: {len(mapped_genes)}')
    
    goea_results_all = goeaobj.run_study(mapped_genes)
    goea_results_sig = [r for r in goea_results_all if r.p_fdr_bh < 0.05]
    GO = pd.DataFrame(list(map(lambda x: [x.GO, x.goterm.name, x.goterm.namespace, x.p_uncorrected, x.p_fdr_bh,\
                   x.ratio_in_study[0], x.ratio_in_study[1], GO_items.count(x.GO), list(map(lambda y: inv_map[y], x.study_items)),\
                   ], goea_results_sig)), columns = ['GO', 'term', 'class', 'p', 'p_corr', 'n_genes',\
                                                    'n_study', 'n_go', 'study_genes'])

    GO = GO[GO.n_genes > 1]
    return GO

In [ ]:
test_genes = scvi_de_upreg_genelist
df = go_it(test_genes)

In [ ]:
df

In [ ]:
#Percent of genes in the GO term that were in the DE genes
#For example: 'GO:0045893' has a total of 441 genes. 77 of them appeared in the data, so % of this GO represented in data = 17%
df['Percent_represented'] = (df.n_genes/df.n_go)*100

In [ ]:
df = df.sort_values('n_genes', ascending=False)
df

In [ ]:
#capitalize the words in the "Term" column to make it look nicer
df.term = df.term.str.capitalize()
df['term'] = df['term'].str.replace('dna','DNA') #one of the terms has 'dna'; make this capitalized
df['term'] = df['term'].str.replace('Camp','cAMP')

In [ ]:
df

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as matplotlib
from matplotlib import cm
import seaborn as sns
import textwrap

In [ ]:
#sns.reset_defaults()

In [ ]:
# #fig, ax = plt.subplots(figsize = (0.5, 2.75))

# plt.figure(figsize = (15,5))
# ax = sns.barplot(data = df, x = 'term', y = 'n_genes', color='red', linewidth=1, edgecolor="black")

# ax.set_xticklabels([textwrap.fill(e, 25) for e in df['term']], fontsize=10) #do text wrapping after n characters

# ax.set_title('GO Pathways of Differentally Expressed Upregulated Genes in UV-B Dosed Samples', fontsize='11')

# ax.set_xlabel('', fontsize='12')
# ax.set_ylabel('Number of Genes', fontsize='12')


# plt.savefig('6_28_24_BoxPlot_ZappedUpregGOTermsBarPlot.png', dpi='figure')
# plt.show()

# #I tried plotting here, but its much easier to do/manipulate in excel-- therefore exported to excel data and plotted there

In [ ]:
df.to_csv('8_29_24_GOTerms_ZappedUpregulated.csv')

# GO Enrichment: Using SCVI Tools to do DE analysis, then using GOATOOLS to assign GO Terms.

In [ ]:
#Differential Expression: Upregulated Genes in Controls
scvi_de2 = ZvC_DE_genes 

#Filter the output dataframe: keep only fdr 0.05=True, and lfc is < -0.5.
scvi_de2 = scvi_de2[(scvi_de2['is_de_fdr_0.05']) & (scvi_de2.lfc_mean < -0.5)] #above looked at all genes with lfc > 0.5 (upregulated in UVB-dosed.) Now looking at all genes with lfc < -0.05 (downregulated in dosed; therefore upregulated in controls)

scvi_de2_upreg_genelist = scvi_de2.Gene_symbol
scvi_de2_upreg_genelist

In [ ]:
scvi_de2

In [ ]:
test_genes2 = scvi_de2_upreg_genelist
df2 = go_it(test_genes2)

df2

In [ ]:
#Percent of genes in the GO term that were in the DE genes
#For example: 'GO:0045893' has a total of 441 genes. 77 of them appeared in the data, so % of this GO represented in data = 17%
df2['Percent_represented'] = (df2.n_genes/df2.n_go)*100

df2 = df2.sort_values('n_genes', ascending=False)
df2

import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib import cm
import seaborn as sns
import textwrap

In [ ]:
#capitalize the words in the "Term" column to make it look nicer
df2.term = df2.term.str.capitalize()
df2['term'] = df2['term'].str.replace('dna','DNA') #one of the terms has 'dna'; make this capitalized

In [ ]:
# #fig, ax = plt.subplots(figsize = (0.5, 2.75))

# plt.figure(figsize = (15,5))
# ax = sns.barplot(data = df2, x = 'term', y = 'n_genes', color='blue', linewidth=1, edgecolor="black")

# ax.set_xticklabels([textwrap.fill(e, 13) for e in df2['term']], fontsize=7) #do text wrapping after n characters

# ax.set_title('GO Pathways of Differentally Expressed Upregulated Genes in Control Samples', fontsize='11')

# ax.set_xlabel('', fontsize='12')
# ax.set_ylabel('Number of Genes', fontsize='12')

# matplotlib.pyplot.savefig('6_28_24_BoxPlot_ControlUpregGOTermsBarPlot.png', dpi='figure')
# plt.show()

# #As with other plot, exported data to excel and plotted there. 

In [ ]:
df2.to_csv('8_29_24_GOTerms_ControlsUpregulated.csv')

#NOTE: these .csv files were saved as excel files. Barplots were redrawn in Excel (better flexibility to edit)

In [ ]:
#To load a previously trained and saved model and adata object

import os
import scanpy as sc
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

import sys
sys.path.append('C:/Users/Riss Kellogg/miniconda3/envs/scvi-env/Lib/site-packages')
import doubletdetection
import scvi

#adata = sc.read_h5ad('6_27_24_cellbenderclean_adata_norm_scaled.h5ad')
adata = sc.read_h5ad('9_6_24_cellbenderclean_adata_norm_scaled_rawdatasaved.h5ad')
model = scvi.model.SCVI.load('6_27_24_cellbender_Ctrl134_Zap134_adatasintegrated.model', adata = adata)


In [ ]:
adata

# Check if adata.X is RAW or NORMALIZED. Use RAW for plotting.

In [ ]:
#What does adata.X look like? If raw data, it should be integers (int). If normalized, it will be floating-point numbers.
adata.X

In [ ]:
print(adata.X[:3, :3])

#So adata.X is all normalized (everything is a decimal point).

In [ ]:
adata
#You can swap adata.X to be adata.layers['counts'] (raw counts) and adata.layers['normalized_counts'] (normalized for UMAP plotting)

# PHAGOCYTES SCVI Differential Expression Analysis do this, save output as dataframe

In [ ]:
#Differential Expression of Phagocytes
PhagocyteDEGs = model.differential_expression(
    idx1 = [(adata.obs['cell_type'].isin(['Phagocytes'])) & (adata.obs.condition == 'UVB-dosed')],
    idx2 = [(adata.obs['cell_type'].isin(['Phagocytes'])) & (adata.obs.condition == 'Control')]
    )

PhagocyteDEGs = PhagocyteDEGs[(PhagocyteDEGs['is_de_fdr_0.05'])] #filter so only DE genes are shown

In [ ]:
#What is the maximum value lfc_mean value in this dataframe?

PhagocyteDEGs.loc[PhagocyteDEGs['lfc_mean'].idxmax()]

In [ ]:
# manipulate the DE results for plotting
PhagocyteDEGs["log10_pscore"] = np.log10(PhagocyteDEGs["proba_not_de"]) #get the log10p value
PhagocyteDEGs["negativelog10_pscore"] = (PhagocyteDEGs["log10_pscore"]*-1) #get the NEGATIVE log10p-value (what is finally plotted)
PhagocyteDEGs = PhagocyteDEGs.join(adata.var, how="inner")

In [ ]:
# #assign an Entrez ID column by copying the index column to it
PhagocyteDEGs["Gene_symbol"] = (PhagocyteDEGs.index)

In [ ]:
#Need to somehow map "gene name" to df.index... 
# # Load lookup table (csv)
lookup = pd.read_excel('SpurpLOClookuptable.xlsx', engine='openpyxl')

In [ ]:
lookup

In [ ]:
# #"lookup" is a pandas dataframe, and "PhagocyteDEGs" is a pandas dataframe.
# #Both contain the column "Gene_symbol", so should be able to merge on that. should end up with a new column "Gene_name"
PhagocyteDEGs = pd.merge(PhagocyteDEGs, lookup, on='Gene_symbol', how = 'inner')

In [ ]:
PhagocyteDEGs.to_csv('PhagocyteDEGs_randomseed5623.csv')

In [ ]:
PhagocyteDEGs.head()

# Load in DEGs as a dataframe

In [ ]:
#Load in dataframe created previously by model.differnetial_expression

PhagocyteDEGs = pd.read_excel('C:/Users/Riss Kellogg/Desktop/scRNAseq Results C134 Z134 6_28_24/PhagocyteDEGs_randomseed5623.xlsx', sheet_name='ALL')
#"C:\Users\Riss Kellogg\Desktop\scRNAseq Results C134 Z134 6_28_24\PhagocyteDEGs_randomseed5623.xlsx"


In [ ]:
PhagocyteDEGs.head()

# Interactive Volcano plot of DE genes using scvi tools

# Bokeh Volcano Plot

In [ ]:
# The dataframe above has what we need to make a bokeh volcano plot: lfc_mean is X, -log10_pscore is Y.
# Import plotting tools
import pandas as pd
import bokeh
import selenium
from bokeh.io import push_notebook, show, output_notebook, export_png, export_svgs
from bokeh.layouts import row
from bokeh.plotting import figure, show, save
from bokeh.transform import factor_cmap
from bokeh.models import ColumnDataSource, Span
from bokeh.models.tools import HoverTool
from bokeh.resources import CDN
from bokeh.embed import file_html

In [ ]:
#Define some Functions

#A function that will allow us to add, color, and outline groups of differentially expressed genes
def add_points(p, df, x, y, color, alpha=0.2, outline=False): #alpha is opacity
    source1 = bokeh.models.ColumnDataSource(df)
    
    p.circle(x=x, y=y, size=7, alpha=alpha, source=source1, color=color, name='circles')
    
    if outline:
        p.circle(x=x, y=y, size=7, alpha=1, source=source1, color='black', fill_color=None, name='outlines')
        
    return p

#A function to separate significantly upregulated and downregulated genes
def selector(df):
    sig_up = (PhagocyteDEGs.negativelog10_pscore > 1.3) & (PhagocyteDEGs.lfc_mean > 1)
    sig_down = (PhagocyteDEGs.negativelog10_pscore > 1.3) & (PhagocyteDEGs.lfc_mean < (-1))
    phagocyte_ups = PhagocyteDEGs[sig_up]
    phagocyte_downs = PhagocyteDEGs[sig_down]
    return phagocyte_ups, phagocyte_downs

In [ ]:
from bokeh.plotting import figure, show

In [ ]:
#Create the volcano plot figure comparing Phagocytes (zap vs control phagocytes)
p = bokeh.plotting.figure(x_range=[-6,6], y_range=[1.22,2.5], width=800, height=800)
p.xaxis.axis_label = 'Average Log2 Fold Change (UVB-Dosed Phagocytes vs. Control Phagocytes)'
p.yaxis.axis_label = '-log10(p value)'

#Add the hover tool
hover = HoverTool()
p.add_tools(hover)
hover.tooltips=[('Gene_ID', '@Gene_ID'),
                ('Name', '@Gene_name'),
                ('LOC', '@Gene_symbol'),
               ('-log10pval', '@negativelog10_pscore'),
               ('lfc', '@lfc_mean')]

#Add the pvalue cutoff of 0.05 line (note, when p=0.05, -log10(0.05) = 1.3)
p.line(x=[-10,10], y=[1.3,1.3],
      color='grey', line_dash='dashed', line_width=1)
from bokeh.models import Label
label = Label(x=-5, y=1.3, text="p = 0.05", text_font_size="16pt", text_color="black")
p.add_layout(label)

#Add the lfc <1 or lfc>1 cutoff lines
p.line(x=[-1,-1], y=[0,10],
      color='grey', line_dash='dashed', line_width=1)
p.line(x=[1,1], y=[0,10], 
      color='grey', line_dash='dashed', line_width=1)

#Add the data
p.circle('lfc_mean','negativelog10_pscore', source=PhagocyteDEGs, fill_alpha=0.1, size=10, color='grey')


#Color sig. upregulated genes in green; Color sig. downregulated genes in red
phagocyte_ups, phagocyte_downs = selector(PhagocyteDEGs)
p = add_points(p, phagocyte_ups, 'lfc_mean','negativelog10_pscore', color='green', alpha=0.6, outline=True)
p = add_points(p, phagocyte_downs, 'lfc_mean', 'negativelog10_pscore', color='red', alpha=0.6, outline=True)

#when using add_points to color ups and downs, it wont accept "legend_label" as an argument, 
#so redoing here so legend labels will show up
p.circle('lfc_mean','negativelog10_pscore', source=phagocyte_ups, color='green', alpha=0.6, 
         legend_label='Significantly upregulated in UVB-dosed Phagocytes')
p.circle('lfc_mean','negativelog10_pscore', source=phagocyte_downs, color='red', alpha=0.6, 
         legend_label='Significantly downregulated in UVB-dosed Phagocytes')


#Add the legend
p.legend.location = "top_right"
#p.legend.title = "Legend"
p.legend.border_line_color = "#000000"
p.legend.border_line_width = 3
p.legend.label_text_font_size = "16pt"
p.legend.label_text_color = "black"

#Adjust axis label settings
p.xaxis.axis_label_text_font_size = "16pt"
p.yaxis.axis_label_text_font_size = "16pt"
p.xaxis.major_label_text_font_size = "16pt"
p.yaxis.major_label_text_font_size = "16pt"
p.xaxis.axis_label_text_color = "black"
p.yaxis.axis_label_text_color = "black"
p.xaxis.axis_label_text_font_style = "normal" #no italics
p.yaxis.axis_label_text_font_style = "normal" #no italics

#Adjust tick label settings
p.xaxis.major_label_text_color = "black"
p.xaxis.major_tick_line_color = "black"
p.yaxis.major_label_text_color = "black"
p.yaxis.major_tick_line_color = "black"

#Keeps the created plot in-line in the notebook
#bokeh.io.reset_output()
#bokeh.io.output_notebook()


from bokeh.io import export_png
from bokeh.io.export import export_svgs
p.output_backend = "svg"

#export_svgs(p, filename='C:/Users/Riss Kellogg/Desktop/scRNAseq Results C134 Z134 6_28_24/INDIVIDUAL FIGURES/Fig2 300dpi/AllCellsDEGs_LFC1_VolcanoPlot.svg')
export_svgs(p, filename='C:/Users/Riss Kellogg/Desktop/scRNAseq Results C134 Z134 6_28_24/INDIVIDUAL FIGURES/Fig3 300dpi/Phagocytes_DEGenes_LFC1_VolcanoPlot.svg')
show(p)

#Green = Significantly upregulated in Zap Phagocytes
#Red = Significantly downregulated in Zap Phagocytes

In [ ]:
len(phagocyte_ups)

In [ ]:
len(phagocyte_downs)

In [ ]:
phagocyte_ups.to_csv('PhagocyteUps.csv')

In [ ]:
phagocyte_downs.to_csv('PhagocyteDowns.csv')

# RED CELLS SCVI Differential Expression Analysis do this, save output as dataframe

In [ ]:
adata.X

In [ ]:
#Differential Expression of Phagocytes
RedCellDEGs = model.differential_expression(
    idx1 = [(adata.obs['cell_type'].isin(['Red Cells'])) & (adata.obs.condition == 'UVB-dosed')],
    idx2 = [(adata.obs['cell_type'].isin(['Red Cells'])) & (adata.obs.condition == 'Control')]
    )

RedCellDEGs = RedCellDEGs[(RedCellDEGs['is_de_fdr_0.05'])] #filter so only DE genes are shown

In [ ]:
#What is the maximum value lfc_mean value in this dataframe?

RedCellDEGs.loc[RedCellDEGs['lfc_mean'].idxmax()]

In [ ]:
# manipulate the DE results for plotting
RedCellDEGs["log10_pscore"] = np.log10(RedCellDEGs["proba_not_de"]) #get the log10p value
RedCellDEGs["negativelog10_pscore"] = (RedCellDEGs["log10_pscore"]*-1) #get the NEGATIVE log10p-value (what is finally plotted)
RedCellDEGs = RedCellDEGs.join(adata.var, how="inner")

In [ ]:
# #assign an Entrez ID column by copying the index column to it
RedCellDEGs["Gene_symbol"] = (RedCellDEGs.index)

In [ ]:
#Need to somehow map "gene name" to df.index... 
# # Load lookup table (csv)
lookup = pd.read_excel('SpurpLOClookuptable.xlsx', engine='openpyxl')

In [ ]:
lookup

In [ ]:
# #"lookup" is a pandas dataframe, and "PhagocyteDEGs" is a pandas dataframe.
# #Both contain the column "Gene_symbol", so should be able to merge on that. should end up with a new column "Gene_name"
RedCellDEGs = pd.merge(RedCellDEGs, lookup, on='Gene_symbol', how = 'inner')

In [ ]:
RedCellDEGs.head()

In [ ]:
RedCellDEGs.to_csv('RedCellDEGs_randomseed5623.csv')

# Load in DEGs as a dataframe

In [ ]:
#Load in dataframe created previously by model.differnetial_expression

RedCellDEGs = pd.read_excel('C:/Users/Riss Kellogg/Desktop/scRNAseq Results C134 Z134 6_28_24/RedCellDEGs_randomseed5623.xlsx', sheet_name='ALL')
#"C:\Users\Riss Kellogg\Desktop\scRNAseq Results C134 Z134 6_28_24\PhagocyteDEGs_randomseed5623.xlsx"


In [ ]:
RedCellDEGs.head()

# Interactive Volcano plot of DE genes using scvi tools

# RED CELLS Bokeh Volcano Plot

In [ ]:
# The dataframe above has what we need to make a bokeh volcano plot: lfc_mean is X, -log10_pscore is Y.
# Import plotting tools
import pandas as pd
import bokeh
import selenium
from bokeh.io import push_notebook, show, output_notebook, export_png, export_svgs
from bokeh.layouts import row
from bokeh.plotting import figure, show, save
from bokeh.transform import factor_cmap
from bokeh.models import ColumnDataSource, Span
from bokeh.models.tools import HoverTool
from bokeh.resources import CDN
from bokeh.embed import file_html

In [ ]:
#Define some Functions

#A function that will allow us to add, color, and outline groups of differentially expressed genes
def add_points(p, df, x, y, color, alpha=0.2, outline=False): #alpha is opacity
    source1 = bokeh.models.ColumnDataSource(df)
    
    p.circle(x=x, y=y, size=7, alpha=alpha, source=source1, color=color, name='circles')
    
    if outline:
        p.circle(x=x, y=y, size=7, alpha=1, source=source1, color='black', fill_color=None, name='outlines')
        
    return p

#A function to separate significantly upregulated and downregulated genes
def selector(df):
    sig_up = (RedCellDEGs.negativelog10_pscore > 1.3) & (RedCellDEGs.lfc_mean > 1)
    sig_down = (RedCellDEGs.negativelog10_pscore > 1.3) & (RedCellDEGs.lfc_mean < (-1))
    reds_ups = RedCellDEGs[sig_up]
    reds_downs = RedCellDEGs[sig_down]
    return reds_ups, reds_downs

In [ ]:
from bokeh.plotting import figure, show

In [ ]:
#Create the volcano plot figure comparing Red Cells (zap vs control red cells)
p = bokeh.plotting.figure(x_range=[-6,6], y_range=[1.22,2.5], width=800, height=800)
p.xaxis.axis_label = 'Average Log2 Fold Change (UVB-Dosed Red Cells vs. Control Red Cells)'
p.yaxis.axis_label = '-log10(p value)'

#Add the hover tool
hover = HoverTool()
p.add_tools(hover)
hover.tooltips=[('Gene_ID', '@Gene_ID'),
                ('Name', '@Gene_name'),
                ('LOC', '@Gene_symbol'),
               ('-log10pval', '@negativelog10_pscore'),
               ('lfc', '@lfc_mean')]

#Add the pvalue cutoff of 0.05 line (note, when p=0.05, -log10(0.05) = 1.3)
p.line(x=[-10,10], y=[1.3,1.3],
      color='grey', line_dash='dashed', line_width=1)
from bokeh.models import Label
label = Label(x=-5, y=1.3, text="p = 0.05", text_font_size="16pt", text_color="black")
p.add_layout(label)

#Add the lfc <1 or lfc>1 cutoff lines
p.line(x=[-1,-1], y=[0,10],
      color='grey', line_dash='dashed', line_width=1)
p.line(x=[1,1], y=[0,10], 
      color='grey', line_dash='dashed', line_width=1)

#Add the data
p.circle('lfc_mean','negativelog10_pscore', source=RedCellDEGs, fill_alpha=0.1, size=10, color='grey')


#Color sig. upregulated genes in green; Color sig. downregulated genes in red
reds_ups, reds_downs = selector(RedCellDEGs)
p = add_points(p, reds_ups, 'lfc_mean','negativelog10_pscore', color='green', alpha=0.6, outline=True)
p = add_points(p, reds_downs, 'lfc_mean', 'negativelog10_pscore', color='red', alpha=0.6, outline=True)

#when using add_points to color ups and downs, it wont accept "legend_label" as an argument, 
#so redoing here so legend labels will show up
p.circle('lfc_mean','negativelog10_pscore', source=reds_ups, color='green', alpha=0.6, 
         legend_label='Significantly upregulated in UVB-dosed Red Cells')
p.circle('lfc_mean','negativelog10_pscore', source=reds_downs, color='red', alpha=0.6, 
         legend_label='Significantly downregulated in UVB-dosed Red Cells')


#Add the legend
p.legend.location = "top_right"
#p.legend.title = "Legend"
p.legend.border_line_color = "#000000"
p.legend.border_line_width = 3
p.legend.label_text_font_size = "16pt"
p.legend.label_text_color = "black"

#Adjust axis label settings
p.xaxis.axis_label_text_font_size = "16pt"
p.yaxis.axis_label_text_font_size = "16pt"
p.xaxis.major_label_text_font_size = "16pt"
p.yaxis.major_label_text_font_size = "16pt"
p.xaxis.axis_label_text_color = "black"
p.yaxis.axis_label_text_color = "black"
p.xaxis.axis_label_text_font_style = "normal" #no italics
p.yaxis.axis_label_text_font_style = "normal" #no italics


#Adjust tick label settings
p.xaxis.major_label_text_color = "black"
p.xaxis.major_tick_line_color = "black"
p.yaxis.major_label_text_color = "black"
p.yaxis.major_tick_line_color = "black"

#Keeps the created plot in-line in the notebook
#bokeh.io.reset_output()
#bokeh.io.output_notebook()


from bokeh.io import export_png
from bokeh.io.export import export_svgs
p.output_backend = "svg"

#export_svgs(p, filename='C:/Users/Riss Kellogg/Desktop/scRNAseq Results C134 Z134 6_28_24/INDIVIDUAL FIGURES/Fig2 300dpi/AllCellsDEGs_LFC1_VolcanoPlot.svg')
export_svgs(p, filename='C:/Users/Riss Kellogg/Desktop/scRNAseq Results C134 Z134 6_28_24/INDIVIDUAL FIGURES/Fig3 300dpi/RedCells_DEGenes_LFC1_VolcanoPlot.svg')
show(p)


#Green = Significantly upregulated in Zap Red Cells
#Red = Significantly downregulated in Zap Red Cells

In [ ]:
len(reds_ups)

In [ ]:
len(reds_downs)

In [ ]:
reds_ups.to_csv('RedsUps.csv')
reds_downs.to_csv('RedsDowns.csv')

# Colorless CELLS SCVI Differential Expression Analysis, save output as dataframe

In [ ]:
adata.X

In [ ]:
#Differential Expression of Phagocytes
ColorlessCellDEGs = model.differential_expression(
    idx1 = [(adata.obs['cell_type'].isin(['Colorless Cells'])) & (adata.obs.condition == 'UVB-dosed')],
    idx2 = [(adata.obs['cell_type'].isin(['Colorless Cells'])) & (adata.obs.condition == 'Control')]
    )

ColorlessCellDEGs = ColorlessCellDEGs[(ColorlessCellDEGs['is_de_fdr_0.05'])] #filter so only DE genes are shown

In [ ]:
#What is the maximum value lfc_mean value in this dataframe?

ColorlessCellDEGs.loc[ColorlessCellDEGs['lfc_mean'].idxmax()]

In [ ]:
# manipulate the DE results for plotting
ColorlessCellDEGs["log10_pscore"] = np.log10(ColorlessCellDEGs["proba_not_de"]) #get the log10p value
ColorlessCellDEGs["negativelog10_pscore"] = (ColorlessCellDEGs["log10_pscore"]*-1) #get the NEGATIVE log10p-value (what is finally plotted)
ColorlessCellDEGs = ColorlessCellDEGs.join(adata.var, how="inner")

In [ ]:
# #assign an Entrez ID column by copying the index column to it
ColorlessCellDEGs["Gene_symbol"] = (ColorlessCellDEGs.index)

In [ ]:
#Need to somehow map "gene name" to df.index... 
# # Load lookup table (csv)
lookup = pd.read_excel('SpurpLOClookuptable.xlsx', engine='openpyxl')

In [ ]:
lookup

In [ ]:
# #"lookup" is a pandas dataframe, and "PhagocyteDEGs" is a pandas dataframe.
# #Both contain the column "Gene_symbol", so should be able to merge on that. should end up with a new column "Gene_name"
ColorlessCellDEGs = pd.merge(ColorlessCellDEGs, lookup, on='Gene_symbol', how = 'inner')

In [ ]:
ColorlessCellDEGs.head()

In [ ]:
ColorlessCellDEGs.to_csv('ColorlessCellDEGs_randomseed5623.csv')

# Load in DEGs as a dataframe

In [ ]:
#Load in dataframe created previously by model.differnetial_expression

ColorlessCellDEGs = pd.read_excel('C:/Users/Riss Kellogg/Desktop/scRNAseq Results C134 Z134 6_28_24/ColorlessCellDEGs_randomseed5623.xlsx', sheet_name='ALL')
#"C:\Users\Riss Kellogg\Desktop\scRNAseq Results C134 Z134 6_28_24\PhagocyteDEGs_randomseed5623.xlsx"


In [ ]:
ColorlessCellDEGs.head()

# Interactive Volcano plot of DE genes using scvi tools

In [ ]:
# The dataframe above has what we need to make a bokeh volcano plot: lfc_mean is X, -log10_pscore is Y.
# Import plotting tools
import pandas as pd
import bokeh
import selenium
from bokeh.io import push_notebook, show, output_notebook, export_png, export_svgs
from bokeh.layouts import row
from bokeh.plotting import figure, show, save
from bokeh.transform import factor_cmap
from bokeh.models import ColumnDataSource, Span
from bokeh.models.tools import HoverTool
from bokeh.resources import CDN
from bokeh.embed import file_html

In [ ]:
#Define some Functions

#A function that will allow us to add, color, and outline groups of differentially expressed genes
def add_points(p, df, x, y, color, alpha=0.2, outline=False): #alpha is opacity
    source1 = bokeh.models.ColumnDataSource(df)
    
    p.circle(x=x, y=y, size=7, alpha=alpha, source=source1, color=color, name='circles')
    
    if outline:
        p.circle(x=x, y=y, size=7, alpha=1, source=source1, color='black', fill_color=None, name='outlines')
        
    return p

#A function to separate significantly upregulated and downregulated genes
def selector(df):
    sig_up = (ColorlessCellDEGs.negativelog10_pscore > 1.3) & (ColorlessCellDEGs.lfc_mean > 1)
    sig_down = (ColorlessCellDEGs.negativelog10_pscore > 1.3) & (ColorlessCellDEGs.lfc_mean < (-1))
    colorless_ups = ColorlessCellDEGs[sig_up]
    colorless_downs = ColorlessCellDEGs[sig_down]
    return colorless_ups, colorless_downs

In [ ]:
from bokeh.plotting import figure, show

In [ ]:
#Create the volcano plot figure comparing colorless Cells (zap vs control white cells)
p = bokeh.plotting.figure(x_range=[-6,6], y_range=[1.22,2.5], width=800, height=800)
p.xaxis.axis_label = 'Average Log2FoldChange(UVB-Dosed Colorless Cells vs. Control Colorless Cells)'
p.yaxis.axis_label = '-log10(p value)'

#Add the hover tool
hover = HoverTool()
p.add_tools(hover)
hover.tooltips=[('Gene_ID', '@Gene_ID'),
                ('Name', '@Gene_name'),
                ('LOC', '@Gene_symbol'),
               ('-log10pval', '@negativelog10_pscore'),
               ('lfc', '@lfc_mean')]

#Add the pvalue cutoff of 0.05 line (note, when p=0.05, -log10(0.05) = 1.3)
p.line(x=[-10,10], y=[1.3,1.3],
      color='grey', line_dash='dashed', line_width=1)
from bokeh.models import Label
label = Label(x=-5, y=1.3, text="p = 0.05", text_font_size="16pt", text_color="black")
p.add_layout(label)

#Add the lfc <1 or lfc>1 cutoff lines
p.line(x=[-1,-1], y=[0,10],
      color='grey', line_dash='dashed', line_width=1)
p.line(x=[1,1], y=[0,10], 
      color='grey', line_dash='dashed', line_width=1)

#Add the data
p.circle('lfc_mean','negativelog10_pscore', source=ColorlessCellDEGs, fill_alpha=0.1, size=10, color='grey')


#Color sig. upregulated genes in green; Color sig. downregulated genes in red
colorless_ups, colorless_downs = selector(ColorlessCellDEGs)
p = add_points(p, colorless_ups, 'lfc_mean','negativelog10_pscore', color='green', alpha=0.6, outline=True)
p = add_points(p, colorless_downs, 'lfc_mean', 'negativelog10_pscore', color='red', alpha=0.6, outline=True)

#when using add_points to color ups and downs, it wont accept "legend_label" as an argument, 
#so redoing here so legend labels will show up
p.circle('lfc_mean','negativelog10_pscore', source=colorless_ups, color='green', alpha=0.6, 
         legend_label='Significantly upregulated in UVB-dosed Colorless Cells')
p.circle('lfc_mean','negativelog10_pscore', source=colorless_downs, color='red', alpha=0.6, 
         legend_label='Significantly downregulated in UVB-dosed Colorless Cells')


#Add the legend
p.legend.location = "top_right"
#p.legend.title = "Legend"
p.legend.border_line_color = "#000000"
p.legend.border_line_width = 3
p.legend.label_text_font_size = "16pt"
p.legend.label_text_color = "black"

#Adjust axis label settings
p.xaxis.axis_label_text_font_size = "16pt"
p.yaxis.axis_label_text_font_size = "16pt"
p.xaxis.major_label_text_font_size = "16pt"
p.yaxis.major_label_text_font_size = "16pt"
p.xaxis.axis_label_text_color = "black"
p.yaxis.axis_label_text_color = "black"
p.xaxis.axis_label_text_font_style = "normal" #no italics
p.yaxis.axis_label_text_font_style = "normal" #no italics


#Adjust tick label settings
p.xaxis.major_label_text_color = "black"
p.xaxis.major_tick_line_color = "black"
p.yaxis.major_label_text_color = "black"
p.yaxis.major_tick_line_color = "black"

#Keeps the created plot in-line in the notebook
#bokeh.io.reset_output()
#bokeh.io.output_notebook()


from bokeh.io import export_png
from bokeh.io.export import export_svgs
p.output_backend = "svg"

#export_svgs(p, filename='C:/Users/Riss Kellogg/Desktop/scRNAseq Results C134 Z134 6_28_24/INDIVIDUAL FIGURES/Fig2 300dpi/AllCellsDEGs_LFC1_VolcanoPlot.svg')
export_svgs(p, filename='C:/Users/Riss Kellogg/Desktop/scRNAseq Results C134 Z134 6_28_24/INDIVIDUAL FIGURES/Fig3 300dpi/ColorlessCells_DEGenes_LFC1_VolcanoPlot.svg')
show(p)


#Green = Significantly upregulated in Zap clear Cells
#Red = Significantly downregulated in Zap clear Cells

In [ ]:
len(colorless_ups)

In [ ]:
len(colorless_downs)

In [ ]:
colorless_ups.to_csv('ColorlessUps.csv')
colorless_downs.to_csv('ColorlessDowns.csv')

# Vibratile CELLS SCVI Differential Expression Analysis, save output as dataframe

In [ ]:
adata.X

In [ ]:
#Differential Expression of Phagocytes
VibCellDEGs = model.differential_expression(
    idx1 = [(adata.obs['cell_type'].isin(['Vibratile Cells'])) & (adata.obs.condition == 'UVB-dosed')],
    idx2 = [(adata.obs['cell_type'].isin(['Vibratile Cells'])) & (adata.obs.condition == 'Control')]
    )

VibCellDEGs = VibCellDEGs[(VibCellDEGs['is_de_fdr_0.05'])] #filter so only DE genes are shown

In [ ]:
#What is the maximum value lfc_mean value in this dataframe?

VibCellDEGs.loc[VibCellDEGs['lfc_mean'].idxmax()]

In [ ]:
# manipulate the DE results for plotting
VibCellDEGs["log10_pscore"] = np.log10(VibCellDEGs["proba_not_de"]) #get the log10p value
VibCellDEGs["negativelog10_pscore"] = (VibCellDEGs["log10_pscore"]*-1) #get the NEGATIVE log10p-value (what is finally plotted)
VibCellDEGs = VibCellDEGs.join(adata.var, how="inner")

In [ ]:
# #assign an Entrez ID column by copying the index column to it
VibCellDEGs["Gene_symbol"] = (VibCellDEGs.index)

In [ ]:
#Need to somehow map "gene name" to df.index... 
# # Load lookup table (csv)
lookup = pd.read_excel('SpurpLOClookuptable.xlsx', engine='openpyxl')

In [ ]:
lookup

In [ ]:
# #"lookup" is a pandas dataframe, and "PhagocyteDEGs" is a pandas dataframe.
# #Both contain the column "Gene_symbol", so should be able to merge on that. should end up with a new column "Gene_name"
VibCellDEGs = pd.merge(VibCellDEGs, lookup, on='Gene_symbol', how = 'inner')

In [ ]:
VibCellDEGs.head()

In [ ]:
VibCellDEGs.to_csv('VibCellDEGs_randomseed5623.csv')

# Load in DEGs as a dataframe

In [ ]:
#Load in dataframe created previously by model.differnetial_expression

VibCellDEGs = pd.read_excel('C:/Users/Riss Kellogg/Desktop/scRNAseq Results C134 Z134 6_28_24/VibCellDEGs_randomseed5623.xlsx', sheet_name='ALL')


In [ ]:
VibCellDEGs.head()

# Interactive Volcano plot of DE genes using scvi tools

In [ ]:
# The dataframe above has what we need to make a bokeh volcano plot: lfc_mean is X, -log10_pscore is Y.
# Import plotting tools
import pandas as pd
import bokeh
import selenium
from bokeh.io import push_notebook, show, output_notebook, export_png, export_svgs
from bokeh.layouts import row
from bokeh.plotting import figure, show, save
from bokeh.transform import factor_cmap
from bokeh.models import ColumnDataSource, Span
from bokeh.models.tools import HoverTool
from bokeh.resources import CDN
from bokeh.embed import file_html

In [ ]:
#Define some Functions

#A function that will allow us to add, color, and outline groups of differentially expressed genes
def add_points(p, df, x, y, color, alpha=0.2, outline=False): #alpha is opacity
    source1 = bokeh.models.ColumnDataSource(df)
    
    p.circle(x=x, y=y, size=7, alpha=alpha, source=source1, color=color, name='circles')
    
    if outline:
        p.circle(x=x, y=y, size=7, alpha=1, source=source1, color='black', fill_color=None, name='outlines')
        
    return p

#A function to separate significantly upregulated and downregulated genes
def selector(df):
    sig_up = (VibCellDEGs.negativelog10_pscore > 1.3) & (VibCellDEGs.lfc_mean > 1)
    sig_down = (VibCellDEGs.negativelog10_pscore > 1.3) & (VibCellDEGs.lfc_mean < (-1))
    vibs_ups = VibCellDEGs[sig_up]
    vibs_downs = VibCellDEGs[sig_down]
    return vibs_ups, vibs_downs

In [ ]:
from bokeh.plotting import figure, show

In [ ]:
#Create the volcano plot figure comparing vib Cells (zap vs control vib cells)
p = bokeh.plotting.figure(x_range=[-6,6], y_range=[1.22,2.5], width=800, height=800)
p.xaxis.axis_label = 'Average Log2FoldChange(UVB-Dosed Vibratile Cells vs. Control Vibratile Cells)'
p.yaxis.axis_label = '-log10(p value)'

#Add the hover tool
hover = HoverTool()
p.add_tools(hover)
hover.tooltips=[('Gene_ID', '@Gene_ID'),
                ('Name', '@Gene_name'),
                ('LOC', '@Gene_symbol'),
               ('-log10pval', '@negativelog10_pscore'),
               ('lfc', '@lfc_mean')]

#Add the pvalue cutoff of 0.05 line (note, when p=0.05, -log10(0.05) = 1.3)
p.line(x=[-10,10], y=[1.3,1.3],
      color='grey', line_dash='dashed', line_width=1)
from bokeh.models import Label
label = Label(x=-5, y=1.3, text="p = 0.05", text_font_size="16pt", text_color="black")
p.add_layout(label)

#Add the lfc <1 or lfc>1 cutoff lines
p.line(x=[-1,-1], y=[0,10],
      color='grey', line_dash='dashed', line_width=1)
p.line(x=[1,1], y=[0,10], 
      color='grey', line_dash='dashed', line_width=1)

#Add the data
p.circle('lfc_mean','negativelog10_pscore', source=VibCellDEGs, fill_alpha=0.1, size=10, color='grey')


#Color sig. upregulated genes in green; Color sig. downregulated genes in red
vibs_ups, vibs_downs = selector(VibCellDEGs)
p = add_points(p, vibs_ups, 'lfc_mean','negativelog10_pscore', color='green', alpha=0.6, outline=True)
p = add_points(p, vibs_downs, 'lfc_mean', 'negativelog10_pscore', color='red', alpha=0.6, outline=True)

#when using add_points to color ups and downs, it wont accept "legend_label" as an argument, 
#so redoing here so legend labels will show up
p.circle('lfc_mean','negativelog10_pscore', source=vibs_ups, color='green', alpha=0.6, 
         legend_label='Significantly upregulated in UVB-dosed Vibratile Cells')
p.circle('lfc_mean','negativelog10_pscore', source=vibs_downs, color='red', alpha=0.6, 
         legend_label='Significantly downregulated in UVB-dosed Vibratile Cells')


#Add the legend
p.legend.location = "top_right"
#p.legend.title = "Legend"
p.legend.border_line_color = "#000000"
p.legend.border_line_width = 3
p.legend.label_text_font_size = "16pt"
p.legend.label_text_color = "black"

#Adjust axis label settings
p.xaxis.axis_label_text_font_size = "16pt"
p.yaxis.axis_label_text_font_size = "16pt"
p.xaxis.major_label_text_font_size = "16pt"
p.yaxis.major_label_text_font_size = "16pt"
p.xaxis.axis_label_text_color = "black"
p.yaxis.axis_label_text_color = "black"
p.xaxis.axis_label_text_font_style = "normal" #no italics
p.yaxis.axis_label_text_font_style = "normal" #no italics


#Adjust tick label settings
p.xaxis.major_label_text_color = "black"
p.xaxis.major_tick_line_color = "black"
p.yaxis.major_label_text_color = "black"
p.yaxis.major_tick_line_color = "black"

#Keeps the created plot in-line in the notebook
#bokeh.io.reset_output()
#bokeh.io.output_notebook()


from bokeh.io import export_png
from bokeh.io.export import export_svgs
p.output_backend = "svg"

#export_svgs(p, filename='C:/Users/Riss Kellogg/Desktop/scRNAseq Results C134 Z134 6_28_24/INDIVIDUAL FIGURES/Fig2 300dpi/AllCellsDEGs_LFC1_VolcanoPlot.svg')
export_svgs(p, filename='C:/Users/Riss Kellogg/Desktop/scRNAseq Results C134 Z134 6_28_24/INDIVIDUAL FIGURES/Fig3 300dpi/Vibratiles_DEGenes_LFC1_VolcanoPlot.svg')
show(p)


#Green = Significantly upregulated in zap vib Cells
#Red = Significantly downregulated in Zap vib Cells

In [ ]:
len(vibs_ups)

In [ ]:
len(vibs_downs)

In [ ]:
vibs_ups.to_csv('VibsUps.csv')
vibs_downs.to_csv('VibsDowns.csv')